# Courbes PDP/ICE par sous-espace SMT — terrainSA

Ce notebook génère les courbes **PDP (Partial Dependence Plot)** et **ICE (Individual
Conditional Expectation)** pour **chacun des 12 sous-espaces** de l'espace de conception
hiérarchique SMT/ADSG utilisé pour les simulations MAELIA sur `terrainSA`.

**Principe.** L'espace SMT est hiérarchique : trois variables de décision catégorielles
(`n_ferti` ∈ {0,1,2,3}, `has_prepa` ∈ {Non, Oui}, `nb_prepa` ∈ {1,2}, cette dernière
n'existant que si `has_prepa = Oui`) définissent **12 sous-espaces**. Dans chaque
sous-espace, ces trois variables sont fixées et seul un sous-ensemble des variables
continues est *actif* (les autres n'ont pas de sens : par exemple une dose de fertilisation
n'existe pas si l'opération correspondante n'a pas lieu).

**Démarche.** À partir du fichier de résultats de simulation `dataset_metamodel.csv`
(10 000 simulations couvrant tout l'espace) :

1. on partitionne les simulations par sous-espace ;
2. pour chaque sous-espace, on entraîne un métamodèle (forêt aléatoire) **sur les seules
   variables continues actives** de ce sous-espace, pour chaque sortie MAELIA ;
3. on trace, pour chaque variable active et chaque sortie, la courbe PDP moyenne
   accompagnée des courbes ICE individuelles.

Les figures sont écrites dans `analysis/pdp_ice_par_sous_espace/`.


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})


In [2]:
# ── Localisation du fichier de résultats de simulation ────────────────────────
# On remonte jusqu'à la racine du dépôt pour être robuste au dossier d'exécution.
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "simulations" / "log_terrainSA" / "dataset_metamodel.csv").exists():
            return p
    return start

REPO = find_repo_root(Path.cwd())
LOG_DIR = REPO / "simulations" / "log_terrainSA"
DATASET = LOG_DIR / "dataset_metamodel.csv"
OUT_DIR = REPO / "analysis" / "pdp_ice_par_sous_espace"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Racine dépôt :", REPO)
print("Dataset      :", DATASET, "|", "trouvé" if DATASET.exists() else "ABSENT")
print("Sorties      :", OUT_DIR)


Racine dépôt : /Users/benjamin/files/Repositories/Sensitivity_analysis_MAELIA
Dataset      : /Users/benjamin/files/Repositories/Sensitivity_analysis_MAELIA/simulations/log_terrainSA/dataset_metamodel.csv | trouvé
Sorties      : /Users/benjamin/files/Repositories/Sensitivity_analysis_MAELIA/analysis/pdp_ice_par_sous_espace


In [3]:
# ── Plan SMT : ordre des 15 paramètres (feat_0 .. feat_14) ────────────────────
# Cet ordre reproduit exactement celui du notebook de simulation
# batch_simulations_smt_terrainSA.ipynb (design_variables de l'AdsgDesignSpaceImpl).
SMT_FEATURES = [
    "n_ferti", "has_prepa", "nb_prepa",           # 0,1,2  variables de décision
    "Date_Semis", "Delta_PREPA_Semis", "Profondeur_Semis",   # 3,4,5
    "Profondeur_Prepa_1", "Profondeur_Prepa_2",   # 6,7
    "Date_F1", "Date_F2", "Date_F3", "Date_Recolte",  # 8,9,10,11
    "Dose_F1", "Dose_F2", "Dose_F3",              # 12,13,14
]
FEAT_COLS = [f"feat_{i}" for i in range(len(SMT_FEATURES))]
RENAME = dict(zip(FEAT_COLS, SMT_FEATURES))

DECISION = ["n_ferti", "has_prepa", "nb_prepa"]
CONTINUOUS = [f for f in SMT_FEATURES if f not in DECISION]
TARGETS = ["N_lixi", "dCorg", "rdt"]

TARGET_LABELS = {
    "N_lixi": "Azote lixivié (N_lixi)",
    "dCorg": "Variation carbone organique (dCorg)",
    "rdt": "Rendement (rdt)",
}
FEATURE_LABELS = {
    "Date_Semis": "Date de semis (j. campagne)",
    "Delta_PREPA_Semis": "Délai préparation→semis (j)",
    "Profondeur_Semis": "Profondeur de semis (cm)",
    "Profondeur_Prepa_1": "Profondeur préparation 1 (cm)",
    "Profondeur_Prepa_2": "Profondeur préparation 2 (cm)",
    "Date_F1": "Date fertilisation 1 (j.)", "Date_F2": "Date fertilisation 2 (j.)",
    "Date_F3": "Date fertilisation 3 (j.)", "Date_Recolte": "Date de récolte (j.)",
    "Dose_F1": "Dose fertilisation 1 (kgN/ha)", "Dose_F2": "Dose fertilisation 2 (kgN/ha)",
    "Dose_F3": "Dose fertilisation 3 (kgN/ha)",
}


In [4]:
# ── Chargement et décodage des variables de décision ──────────────────────────
raw = pd.read_csv(DATASET)
df = raw.rename(columns=RENAME).copy()

# n_ferti est directement 0..3 ; has_prepa et nb_prepa sont des index ordinaux 0/1.
df["n_ferti"] = df["n_ferti"].round().astype(int)
df["has_prepa"] = df["has_prepa"].round().astype(int)          # 0 = Non, 1 = Oui
df["nb_prepa_raw"] = df["nb_prepa"].round().astype(int)         # 0 = 1 prépa, 1 = 2 prépas
# nb_prepa n'a de sens que si has_prepa == 1
df["nb_prepa"] = np.where(df["has_prepa"] == 1, df["nb_prepa_raw"] + 1, 0)

print("Simulations chargées :", len(df))
print(df.groupby(["n_ferti", "has_prepa", "nb_prepa"]).size().rename("n"))


Simulations chargées : 10000
n_ferti  has_prepa  nb_prepa
0        0          0           1211
         1          1            631
                    2            621
1        0          0           1312
         1          1            647
                    2            623
2        0          0           1219
         1          1            628
                    2            633
3        0          0           1208
         1          1            649
                    2            618
Name: n, dtype: int64


In [5]:
# ── Règle d'activation des variables continues par sous-espace ────────────────
def active_continuous(n_ferti: int, has_prepa: int, nb_prepa: int) -> list[str]:
    """Variables continues actives pour un sous-espace donné."""
    feats = ["Date_Semis", "Profondeur_Semis", "Date_Recolte"]  # toujours actives
    if has_prepa == 1:
        feats += ["Delta_PREPA_Semis", "Profondeur_Prepa_1"]
        if nb_prepa == 2:
            feats += ["Profondeur_Prepa_2"]
    if n_ferti >= 1:
        feats += ["Date_F1", "Dose_F1"]
    if n_ferti >= 2:
        feats += ["Date_F2", "Dose_F2"]
    if n_ferti >= 3:
        feats += ["Date_F3", "Dose_F3"]
    # ordre stable selon SMT_FEATURES
    return [f for f in CONTINUOUS if f in feats]


def subspace_label(n_ferti, has_prepa, nb_prepa) -> str:
    prep = "sansPrepa" if has_prepa == 0 else f"prepa{nb_prepa}"
    return f"nferti{n_ferti}_{prep}"


# Énumération des 12 sous-espaces valides
SUBSPACES = []
for nf in [0, 1, 2, 3]:
    SUBSPACES.append((nf, 0, 0))          # sans préparation
    SUBSPACES.append((nf, 1, 1))          # préparation, 1 reprise
    SUBSPACES.append((nf, 1, 2))          # préparation, 2 reprises
assert len(SUBSPACES) == 12
for s in SUBSPACES:
    print(subspace_label(*s), "→ actives :", active_continuous(*s))


nferti0_sansPrepa → actives : ['Date_Semis', 'Profondeur_Semis', 'Date_Recolte']
nferti0_prepa1 → actives : ['Date_Semis', 'Delta_PREPA_Semis', 'Profondeur_Semis', 'Profondeur_Prepa_1', 'Date_Recolte']
nferti0_prepa2 → actives : ['Date_Semis', 'Delta_PREPA_Semis', 'Profondeur_Semis', 'Profondeur_Prepa_1', 'Profondeur_Prepa_2', 'Date_Recolte']
nferti1_sansPrepa → actives : ['Date_Semis', 'Profondeur_Semis', 'Date_F1', 'Date_Recolte', 'Dose_F1']
nferti1_prepa1 → actives : ['Date_Semis', 'Delta_PREPA_Semis', 'Profondeur_Semis', 'Profondeur_Prepa_1', 'Date_F1', 'Date_Recolte', 'Dose_F1']
nferti1_prepa2 → actives : ['Date_Semis', 'Delta_PREPA_Semis', 'Profondeur_Semis', 'Profondeur_Prepa_1', 'Profondeur_Prepa_2', 'Date_F1', 'Date_Recolte', 'Dose_F1']
nferti2_sansPrepa → actives : ['Date_Semis', 'Profondeur_Semis', 'Date_F1', 'Date_F2', 'Date_Recolte', 'Dose_F1', 'Dose_F2']
nferti2_prepa1 → actives : ['Date_Semis', 'Delta_PREPA_Semis', 'Profondeur_Semis', 'Profondeur_Prepa_1', 'Date_F1', 'Da

In [6]:
# ── Fonction PDP/ICE ──────────────────────────────────────────────────────────
def pdp_ice_curve(model, X, feature, grid_size=15, ice_sample=100, seed=42):
    """Retourne (grille, matrice ICE [n_ice x grid], PDP moyenne)."""
    rng = np.random.default_rng(seed)
    lo, hi = np.quantile(X[feature], [0.02, 0.98])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = X[feature].min(), X[feature].max()
    grid = np.linspace(lo, hi, grid_size)
    n = min(ice_sample, len(X))
    idx = rng.choice(len(X), size=n, replace=False)
    X_ref = X.iloc[idx].reset_index(drop=True)
    ice = np.empty((n, grid_size))
    for j, v in enumerate(grid):
        X_mod = X_ref.copy()
        X_mod[feature] = v
        ice[:, j] = model.predict(X_mod)
    return grid, ice, ice.mean(axis=0)


def plot_subspace_target(sub, target, model, X, features, q2, out_path):
    """Une figure : une sous-figure PDP/ICE par variable active."""
    ncol = min(3, len(features))
    nrow = int(np.ceil(len(features) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.0 * ncol, 3.7 * nrow), squeeze=False)
    for k, feat in enumerate(features):
        ax = axes[k // ncol][k % ncol]
        grid, ice, pdp = pdp_ice_curve(model, X, feat)
        for row in ice:
            ax.plot(grid, row, color="#9AB3D4", alpha=0.15, linewidth=0.8)
        ax.plot(grid, pdp, color="#0B3D91", linewidth=3.0, label="PDP moyenne")
        ax.scatter(grid, pdp, color="#0B3D91", s=18, zorder=3)
        ax.set_xlabel(FEATURE_LABELS.get(feat, feat), fontsize=9)
        ax.set_ylabel(target, fontsize=9)
        ax.tick_params(labelsize=8)
    for k in range(len(features), nrow * ncol):
        axes[k // ncol][k % ncol].axis("off")
    nf, hp, npr = sub
    prep = "sans préparation" if hp == 0 else f"préparation ({npr} reprise{'s' if npr == 2 else ''})"
    fig.suptitle(
        f"PDP/ICE — {TARGET_LABELS.get(target, target)}\n"
        f"Sous-espace : n_ferti={nf}, {prep}   |   {len(X)} simulations   |   Q²(test)={q2:.2f}",
        fontsize=12, fontweight="bold",
    )
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


In [7]:
# ── Boucle principale : un métamodèle par (sous-espace, sortie) ───────────────
summary = []
for sub in SUBSPACES:
    nf, hp, npr = sub
    sel = (df["n_ferti"] == nf) & (df["has_prepa"] == hp)
    sel &= (df["nb_prepa"] == npr) if hp == 1 else True
    sub_df = df[sel]
    feats = active_continuous(*sub)
    label = subspace_label(*sub)
    sub_out = OUT_DIR / label
    sub_out.mkdir(parents=True, exist_ok=True)

    for target in TARGETS:
        data = sub_df[feats + [target]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(data) < 40:
            summary.append({"sous_espace": label, "sortie": target,
                            "n": len(data), "q2": np.nan, "statut": "trop peu de points"})
            continue
        X, y = data[feats], data[target]
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
        model = RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                      random_state=42, n_jobs=-1).fit(X_tr, y_tr)
        q2 = r2_score(y_te, model.predict(X_te))
        out_path = sub_out / f"pdp_ice_{label}_{target}.png"
        plot_subspace_target(sub, target, model, X, feats, q2, out_path)
        summary.append({"sous_espace": label, "sortie": target,
                        "n": len(data), "q2": round(q2, 3), "statut": "ok"})

summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT_DIR / "pdp_ice_synthese.csv", index=False)
print("Figures générées dans :", OUT_DIR)
summary_df


Figures générées dans : /Users/benjamin/files/Repositories/Sensitivity_analysis_MAELIA/analysis/pdp_ice_par_sous_espace


,sous_espace,sortie,n,q2,statut
0,nferti0_sansPrepa,N_lixi,1211,-0.089,ok
1,nferti0_sansPrepa,dCorg,1211,-0.119,ok
2,nferti0_sansPrepa,rdt,1211,-0.076,ok
3,nferti0_prepa1,N_lixi,631,0.009,ok
4,nferti0_prepa1,dCorg,631,-0.031,ok
5,nferti0_prepa1,rdt,631,-0.002,ok
6,nferti0_prepa2,N_lixi,621,-0.193,ok
7,nferti0_prepa2,dCorg,621,-0.174,ok
8,nferti0_prepa2,rdt,621,-0.104,ok
9,nferti1_sansPrepa,N_lixi,1312,-0.103,ok


## Synthèse

Le tableau `pdp_ice_synthese.csv` récapitule, pour chaque couple (sous-espace, sortie), le
nombre de simulations utilisées et le Q² du métamodèle sur l'échantillon de test. Les
figures PDP/ICE sont organisées par sous-espace dans `analysis/pdp_ice_par_sous_espace/`.

**Lecture des courbes.** Chaque ligne bleu clair est un scénario individuel (courbe ICE) :
on y fait varier une seule variable active en gardant les autres fixées à leurs valeurs
observées. La ligne bleu foncé est la moyenne de ces courbes (PDP), soit l'effet marginal
moyen de la variable sur la sortie, tel que capté par le métamodèle du sous-espace. Un
faisceau ICE dispersé autour de la PDP signale des interactions avec les autres variables
actives du sous-espace.
